# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.org/) library. It demonstrates how to load metadata, review record sets, extract and process data, and visualize results.

### Dataset Source
The dataset source is provided via a Croissant schema URL ([https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata.to_json()
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])

## 2. Data Overview

Review available record sets, fields, and their IDs. The Croissant schema organizes data into record sets with fields and columns, each uniquely referenced by their `@id` values.

In [ ]:
# List all record sets in the dataset, referencing them by their '@id'
record_sets = []
for rs in dataset.metadata.record_sets():
    record_sets.append(rs['@id'])
    print(f"Record Set @id: {rs['@id']}, Name: {rs.get('name', '<no name>')}")
    # List available fields (columns) for each record set
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field @id: {field['@id']} | Name: {field.get('name', '<no name>')} | DataType: {field.get('dataType', '<no type>')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set @ids (from the overview cell)
record_sets_ids = record_sets

# Create a dictionary mapping each record set @id to a DataFrame
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for Record Set {record_set_id}:", df.columns.tolist())
        print(df.head())
    else:
        print(f"No records found for Record Set {record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes. All entities are referenced by their `@id`.

In [ ]:
# For demonstration, select the first non-empty record set for EDA
selected_record_set_id = None
for rsid in dataframes:
    if not dataframes[rsid].empty:
        selected_record_set_id = rsid
        break

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]

    # Identify a numeric field (column) by its @id and name
    numeric_field_id = None
    for rs in dataset.metadata.record_sets():
        if rs['@id'] == selected_record_set_id and 'fields' in rs:
            for field in rs['fields']:
                # Try to pick a numeric field (Integer or Float dataType)
                if field.get('dataType') in ['schema:Integer', 'schema:Float', 'Integer', 'Float']:
                    numeric_field_id = field['@id']
                    print(f"Using numeric field @id: {numeric_field_id} (Name: {field.get('name', '')})")
                    break
            break

    # If not found, use the first column
    if numeric_field_id is None:
        numeric_field_id = df.columns[0]
        print(f"No explicitly numeric field found, using the first column: {numeric_field_id}")

    # If column exists and is numeric, apply EDA
    if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()  # use mean as threshold example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by any categorical field
        group_field_id = None
        for rs in dataset.metadata.record_sets():
            if rs['@id'] == selected_record_set_id and 'fields' in rs:
                for field in rs['fields']:
                    # Pick a Text/String field for grouping
                    if field.get('dataType') in ['schema:Text', 'schema:String', 'Text', 'String']:
                        group_field_id = field['@id']
                        print(f"Grouping by field @id: {group_field_id} (Name: {field.get('name', '')})")
                        break
                break

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Column {numeric_field_id} is not numeric or doesn't exist for EDA.")
else:
    print("No suitable record set with data found for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Refer to fields and groups by their `@id`.

In [ ]:
# Visualization example: histogram of selected numeric field
if selected_record_set_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Record Set {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show group bar plot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} in {selected_record_set_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrates how to access, extract, and analyze the FAIR^2 dataset using the `mlcroissant` library. By referencing entities via their `@id`, you ensure reproducibility and clarity when working with Croissant schema-based datasets.

Feel free to extend the notebook with more advanced analyses, careful field selection, and domain-specific questions.